In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

### Create Spark Session

In [2]:
spark = SparkSession.builder.appName("basic").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 21:24:16 WARN Utils: Your hostname, musleh, resolves to a loopback address: 127.0.1.1; using 192.168.0.110 instead (on interface enp3s0)
26/09/21 21:24:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 21:24:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Loads demo data

In [4]:
customers = spark.read.option("header", True).csv("../data/raw/customers.csv")

In [5]:
orders = spark.read.option("header", True).csv("../data/raw/orders.csv")

In [6]:
products = spark.read.option("header", True).csv("../data/raw/products.csv")

In [3]:
data = [
    (1, "Alice", "Bangladesh", 25),
    (2, "Bob", "United States", 34),
    (3, "Charlie", "Bangladesh", 42),
]

In [4]:
df = spark.createDataFrame(data, ["customer_id", "name", "country", "age"])

In [5]:
df.show()

+-----------+-------+-------------+---+
|customer_id|   name|      country|age|
+-----------+-------+-------------+---+
|          1|  Alice|   Bangladesh| 25|
|          2|    Bob|United States| 34|
|          3|Charlie|   Bangladesh| 42|
+-----------+-------+-------------+---+



In [6]:
df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- age: long (nullable = true)



In [7]:
df.columns

['customer_id', 'name', 'country', 'age']

In [8]:
df.dtypes

[('customer_id', 'bigint'),
 ('name', 'string'),
 ('country', 'string'),
 ('age', 'bigint')]

In [10]:
df.select("name", (F.col("age") + 1).alias("next_year_age")).show()

+-------+-------------+
|   name|next_year_age|
+-------+-------------+
|  Alice|           26|
|    Bob|           35|
|Charlie|           43|
+-------+-------------+



In [11]:
df.filter((F.col("age") > 30) & (F.col("country") == "Bangladesh")).show()

+-----------+-------+----------+---+
|customer_id|   name|   country|age|
+-----------+-------+----------+---+
|          3|Charlie|Bangladesh| 42|
+-----------+-------+----------+---+



In [14]:
df.select("country").distinct().show()

+-------------+
|      country|
+-------------+
|   Bangladesh|
|United States|
+-------------+



In [17]:
df.groupBy("country").agg(
    F.count("*").alias("customer_count"),
    F.sum("age").alias("total_age"),
    F.max("age").alias("max_age"),
    F.min("age").alias("min_age"),
).show()

+-------------+--------------+---------+-------+-------+
|      country|customer_count|total_age|max_age|min_age|
+-------------+--------------+---------+-------+-------+
|   Bangladesh|             2|       67|     42|     25|
|United States|             1|       34|     34|     34|
+-------------+--------------+---------+-------+-------+



In [18]:
df.createOrReplaceTempView("customers")

In [21]:
spark.sql("""
    SELECT
      country,
      COUNT(*) AS customer_count,
      SUM(age) AS total_age
    FROM customers
    GROUP BY country
""").show()

+-------------+--------------+---------+
|      country|customer_count|total_age|
+-------------+--------------+---------+
|   Bangladesh|             2|       67|
|United States|             1|       34|
+-------------+--------------+---------+



### Window Functions

In [8]:
window = Window.partitionBy("country").orderBy(F.col("total_spent").desc())
result = customers.withColumn("rank", F.row_number().over(window)).filter(
    F.col("rank") <= 3
)
result.show()

+-----------+---------+----------+---+-----------+----+
|customer_id|     name|   country|age|total_spent|rank|
+-----------+---------+----------+---+-----------+----+
|        278| Consalve|Bangladesh| 28|       9906|   1|
|         89|  Yoshiko|Bangladesh| 34|       9869|   2|
|        637|    Ethyl|Bangladesh| 23|        984|   3|
|        664|     Anne|    Bhutan| 63|       7951|   1|
|         86|   Kathie|    Bhutan| 33|       7615|   2|
|        905|    Kelly|    Bhutan| 59|       6643|   3|
|        700|   Adrien|      Iran| 36|       9965|   1|
|        726|    Marin|      Iran| 65|       9955|   2|
|        196|    Butch|      Iran| 54|        993|   3|
|        989|  Ceciley|      Iraq| 39|       9857|   1|
|        939|  Arabele|      Iraq| 33|       9808|   2|
|         26|     Fara|      Iraq| 58|       9308|   3|
|        353|     Zita|     Nepal| 56|       9900|   1|
|        907|     Noni|     Nepal| 49|       9774|   2|
|        982|     Abbe|     Nepal| 20|       975

### UDF

In [21]:
def classify_age(age) -> str:
    if isinstance(age, str):
        age = int(age)
    if not age:
        return "unknown"
    if age < 18:
        return "minor"
    if age < 60:
        return "adult"
    return "senior"

In [22]:
classify_age_udf = F.udf(classify_age, T.StringType())

/home/musleh/programming/python/tutorials/pyspark-learning/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(
/home/musleh/programming/python/tutorials/pyspark-learning/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:135: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [23]:
customers.withColumn("age_group", classify_age_udf(F.col("age"))).show()

+-----------+----------+----------+---+-----------+---------+
|customer_id|      name|   country|age|total_spent|age_group|
+-----------+----------+----------+---+-----------+---------+
|          1|      Yule|  Pakistan| 39|       4431|    adult|
|          2|   Emmalee|  Pakistan| 31|       6974|    adult|
|          3|   Angelle|     Nepal| 40|       4905|    adult|
|          4|   Osbourn|      Iran| 33|       4861|    adult|
|          5|     Trixy|     Nepal| 25|       5316|    adult|
|          6|     Orton|      Iran| 61|       1189|   senior|
|          7|     Mable|      Iran| 18|       8782|    adult|
|          8|    Zorine|      Iran| 58|       1622|    adult|
|          9|   Janette|     Nepal| 60|       4252|   senior|
|         10|  Sapphire|  Pakistan| 69|       9982|   senior|
|         11|    Allsun|      Iran| 51|        692|    adult|
|         12|    Fletch|  Pakistan| 39|       5287|    adult|
|         13|    Giorgi|      Iran| 48|       7205|    adult|
|       

In [17]:
customers.withColumn(
    "age_group",
    F.when(F.col("age") < 18, "minor")
    .when(F.col("age") < 60, "adult")
    .otherwise("senior"),
).show()

+-----------+----------+----------+---+-----------+---------+
|customer_id|      name|   country|age|total_spent|age_group|
+-----------+----------+----------+---+-----------+---------+
|          1|      Yule|  Pakistan| 39|       4431|    adult|
|          2|   Emmalee|  Pakistan| 31|       6974|    adult|
|          3|   Angelle|     Nepal| 40|       4905|    adult|
|          4|   Osbourn|      Iran| 33|       4861|    adult|
|          5|     Trixy|     Nepal| 25|       5316|    adult|
|          6|     Orton|      Iran| 61|       1189|   senior|
|          7|     Mable|      Iran| 18|       8782|    adult|
|          8|    Zorine|      Iran| 58|       1622|    adult|
|          9|   Janette|     Nepal| 60|       4252|   senior|
|         10|  Sapphire|  Pakistan| 69|       9982|   senior|
|         11|    Allsun|      Iran| 51|        692|    adult|
|         12|    Fletch|  Pakistan| 39|       5287|    adult|
|         13|    Giorgi|      Iran| 48|       7205|    adult|
|       